# 05 - Integrated Pipeline

Unified run flow returning predicted rating, predicted cuisine labels, and top recommendations.

Run order after clean kernel: 01 -> 02 -> 03 -> 04 -> 05.

This notebook retrains lightweight task models to provide one callable interface.

In [6]:
# Shared setup
import warnings
import random
import numpy as np
import pandas as pd

from scipy.sparse import hstack
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, precision_recall_fscore_support, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings('ignore')

print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('seed:', SEED)

pandas: 2.2.3
numpy: 2.0.2
seed: 42


In [7]:
DATA_PATH = 'Dataset .csv'
df_raw = pd.read_csv(DATA_PATH)

EXPECTED_COLUMNS = [
    'Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address',
    'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines',
    'Average Cost for two', 'Currency', 'Has Table booking',
    'Has Online delivery', 'Is delivering now', 'Switch to order menu',
    'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes'
]
missing = sorted(set(EXPECTED_COLUMNS) - set(df_raw.columns))
assert not missing, f'Missing required columns: {missing}'
print('Loaded shape:', df_raw.shape)

Loaded shape: (9551, 21)


In [8]:
UNIVERSAL_DROP_COLUMNS = [
    'Restaurant ID', 'Address', 'Locality', 'Locality Verbose',
    'Longitude', 'Latitude', 'Switch to order menu',
    'Rating color', 'Rating text'
 ]
YES_NO_COLUMNS = ['Has Table booking', 'Has Online delivery', 'Is delivering now']

def map_yes_no(series: pd.Series) -> pd.Series:
    mapping = {'Yes': 1, 'No': 0}
    return series.map(mapping).fillna(series)

def cuisine_tokens(value):
    if pd.isna(value):
        return ['Unknown']
    tokens = [x.strip() for x in str(value).split(',') if x.strip()]
    return tokens if tokens else ['Unknown']

def apply_shared_preprocessing(df: pd.DataFrame, task: str) -> pd.DataFrame:
    out = df.copy()

    for c in YES_NO_COLUMNS:
        if c in out.columns:
            out[c] = map_yes_no(out[c])

    if task in {'task1', 'task2'}:
        out['Cuisines'] = out['Cuisines'].fillna('Unknown')
    elif task == 'task3':
        out = out.dropna(subset=['Cuisines']).copy()

    out['Cuisine Tokens'] = out['Cuisines'].apply(cuisine_tokens)
    out['Primary Cuisine'] = out['Cuisine Tokens'].apply(lambda xs: xs[0] if xs else 'Unknown')
    out['Cuisine Text'] = out['Cuisine Tokens'].apply(lambda xs: ' '.join(xs).lower())

    drop_cols = [c for c in UNIVERSAL_DROP_COLUMNS if c in out.columns]
    out = out.drop(columns=drop_cols)
    return out

In [9]:
# Build task-specific frames
df_task1 = apply_shared_preprocessing(df_raw, task='task1')
df_task2 = apply_shared_preprocessing(df_raw, task='task2')
df_task3 = apply_shared_preprocessing(df_raw, task='task3')

# Anti-leakage assertions
forbidden_features = ['Rating color', 'Rating text']
df_task1 = df_task1.drop(columns=forbidden_features, errors='ignore')
df_task2 = df_task2.drop(columns=forbidden_features, errors='ignore')
df_task3 = df_task3.drop(columns=forbidden_features, errors='ignore')
assert not any(c in df_task1.columns for c in forbidden_features)
assert not any(c in df_task2.columns for c in forbidden_features)
assert not any(c in df_task3.columns for c in forbidden_features)
print('Anti-leakage assertions passed in integrated notebook.')

Anti-leakage assertions passed in integrated notebook.


In [10]:
# Train rating model (Task 1)
rating_features = [
    'Country Code', 'City', 'Primary Cuisine', 'Average Cost for two', 'Currency',
    'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Price range', 'Votes'
]
Xr = df_task1[rating_features]
yr = df_task1['Aggregate rating']

cat_r = ['City', 'Primary Cuisine', 'Currency']
num_r = [
    'Country Code', 'Average Cost for two', 'Has Table booking',
    'Has Online delivery', 'Is delivering now', 'Price range', 'Votes'
]

rating_model = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_r),
        ('num', 'passthrough', num_r)
    ])),
    ('model', DecisionTreeRegressor(random_state=SEED, max_depth=12, min_samples_leaf=5))
])
rating_model.fit(Xr, yr)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['City', 'Primary Cuisine',
                                                   'Currency']),
                                                 ('num', 'passthrough',
                                                  ['Country Code',
                                                   'Average Cost for two',
                                                   'Has Table booking',
                                                   'Has Online delivery',
                                                   'Is delivering now',
                                                   'Price range', 'Votes'])])),
                ('model',
                 DecisionTreeRegressor(max_depth=12, min_samples_leaf=5,
                                       random_state=42))])

In [11]:
# Train cuisine model (Task 3)
label_lists = df_task3['Cuisine Tokens']
min_support = 20
label_counts = pd.Series([l for ls in label_lists for l in ls]).value_counts()
kept = set(label_counts[label_counts >= min_support].index)
label_lists = label_lists.apply(lambda ls: [x for x in ls if x in kept])
mask = label_lists.apply(len) > 0
df_task3f = df_task3[mask].copy()
label_lists = label_lists[mask]

mlb = MultiLabelBinarizer()
Yc = mlb.fit_transform(label_lists)

cuisine_features = [
    'Country Code', 'City', 'Average Cost for two', 'Currency',
    'Has Table booking', 'Has Online delivery', 'Is delivering now',
    'Price range', 'Votes', 'Aggregate rating'
]
Xc = df_task3f[cuisine_features]

cat_c = ['City', 'Currency']
num_c = [
    'Country Code', 'Average Cost for two', 'Has Table booking',
    'Has Online delivery', 'Is delivering now', 'Price range', 'Votes', 'Aggregate rating'
]

cuisine_model = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_c),
        ('num', 'passthrough', num_c)
    ])),
    ('clf', OneVsRestClassifier(LogisticRegression(max_iter=2000, class_weight='balanced')))
])
cuisine_model.fit(Xc, Yc)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['City', 'Currency']),
                                                 ('num', 'passthrough',
                                                  ['Country Code',
                                                   'Average Cost for two',
                                                   'Has Table booking',
                                                   'Has Online delivery',
                                                   'Is delivering now',
                                                   'Price range', 'Votes',
                                                   'Aggregate rating'])])),
                ('clf',
                 OneVsRestClassifier(estimator=LogisticRegression(class_weight='balanced',
                                                                  max_iter=2000)))])

In [12]:
# Build recommender artifacts (Task 2)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
cuisine_matrix = vectorizer.fit_transform(df_task2['Cuisine Text'])
num_scaled = MinMaxScaler().fit_transform(df_task2[['Price range', 'Aggregate rating']].fillna(0).values)
content_matrix = hstack([cuisine_matrix, num_scaled])
sim_matrix = cosine_similarity(content_matrix, dense_output=False)

name_to_indices = df_task2.groupby('Restaurant Name').apply(lambda x: list(x.index)).to_dict()

def resolve_index(name: str) -> int:
    idxs = name_to_indices.get(name)
    if not idxs:
        raise ValueError(f'Restaurant not found: {name}')
    if len(idxs) == 1:
        return idxs[0]
    return int(df_task2.loc[idxs].sort_values('Votes', ascending=False).index[0])

def recommend(name: str, top_n: int = 5) -> pd.DataFrame:
    idx = resolve_index(name)
    sims = sim_matrix[idx].toarray().ravel()
    cand = df_task2.copy()
    cand['similarity'] = sims
    cand = cand[cand.index != idx]
    cand = cand[cand['Aggregate rating'] > 0]
    cand = cand.sort_values(['similarity', 'Votes'], ascending=[False, False])
    cols = ['Restaurant Name', 'City', 'Cuisines', 'Price range', 'Aggregate rating', 'Votes', 'similarity']
    return cand[cols].head(top_n).reset_index(drop=True)

In [13]:
def infer_cuisines(row_df: pd.DataFrame, threshold: float = 0.35):
    probs = cuisine_model.predict_proba(row_df[cuisine_features])[0]
    labels = [mlb.classes_[i] for i, p in enumerate(probs) if p >= threshold]
    if not labels:
        labels = [mlb.classes_[int(np.argmax(probs))]]
    return labels

def run_flow(query_name: str = None, manual_profile: dict = None, top_n: int = 5) -> dict:
    if query_name is None and manual_profile is None:
        raise ValueError('Provide either query_name or manual_profile.')

    if query_name is not None:
        idx = resolve_index(query_name)
        row = df_task1.loc[[idx]].copy()
        source_name = df_task2.loc[idx, 'Restaurant Name']
        recs = recommend(source_name, top_n=top_n)
    else:
        defaults = {
            'Country Code': int(df_task1['Country Code'].mode()[0]),
            'City': str(df_task1['City'].mode()[0]),
            'Primary Cuisine': 'Unknown',
            'Average Cost for two': float(df_task1['Average Cost for two'].median()),
            'Currency': str(df_task1['Currency'].mode()[0]),
            'Has Table booking': 0,
            'Has Online delivery': 0,
            'Is delivering now': 0,
            'Price range': int(df_task1['Price range'].median()),
            'Votes': float(df_task1['Votes'].median()),
            'Aggregate rating': float(df_task1['Aggregate rating'].median())
        }
        defaults.update(manual_profile or {})

        row = pd.DataFrame([defaults])[rating_features].copy()
        source_name = None

        # Recommendation for manual profile uses heuristic matching.
        proxy = defaults.get('Primary Cuisine', 'Unknown')
        temp = df_task2.copy()
        temp['score'] = (
            temp['Cuisines'].str.contains(str(proxy), case=False, na=False).astype(int) * 0.7
            + (1 - (temp['Price range'] - defaults['Price range']).abs() / 4) * 0.2
            + (temp['Aggregate rating'] / 5.0) * 0.1
        )
        temp = temp[temp['Aggregate rating'] > 0].sort_values(['score', 'Votes'], ascending=[False, False])
        recs = temp[['Restaurant Name', 'City', 'Cuisines', 'Price range', 'Aggregate rating', 'Votes', 'score']].head(top_n).reset_index(drop=True)

    pred_rating = float(rating_model.predict(row)[0])

    cuisine_row = pd.DataFrame([{
        'Country Code': int(row['Country Code'].iloc[0]),
        'City': str(row['City'].iloc[0]),
        'Average Cost for two': float(row['Average Cost for two'].iloc[0]),
        'Currency': str(row['Currency'].iloc[0]),
        'Has Table booking': int(row['Has Table booking'].iloc[0]),
        'Has Online delivery': int(row['Has Online delivery'].iloc[0]),
        'Is delivering now': int(row['Is delivering now'].iloc[0]),
        'Price range': int(row['Price range'].iloc[0]),
        'Votes': float(row['Votes'].iloc[0]),
        'Aggregate rating': pred_rating
    }])

    pred_cuisines = infer_cuisines(cuisine_row)

    return {
        'source_restaurant': source_name,
        'predicted_rating': round(pred_rating, 3),
        'predicted_cuisine_labels': pred_cuisines,
        'top_recommendations': recs
    }

In [14]:
# Demo 1: known restaurant query
demo_name = df_task2.sort_values('Votes', ascending=False)['Restaurant Name'].iloc[0]
demo1 = run_flow(query_name=demo_name, top_n=5)
print('Source:', demo1['source_restaurant'])
print('Predicted rating:', demo1['predicted_rating'])
print('Predicted cuisines:', demo1['predicted_cuisine_labels'])
display(demo1['top_recommendations'])

Source: Toit
Predicted rating: 4.671
Predicted cuisines: ['American', 'Asian', 'Bengali', 'Burger', 'Cafe', 'Continental', 'Desserts', 'European', 'Finger Food', 'French', 'Italian', 'Lebanese', 'Mediterranean', 'Mexican', 'Mughlai', 'North Indian', 'Pizza', 'Rajasthani', 'Seafood', 'South Indian']


,Restaurant Name,City,Cuisines,Price range,Aggregate rating,Votes,similarity
0,Vintage Machine,Lucknow,"Italian, American, Lebanese",3,4.3,887,0.836922
1,AMPM Caf�� & Bar,New Delhi,"Continental, Italian, American",3,4.1,1980,0.827163
2,Portneuf Valley Brewing,Pocatello,"American, Pizza",2,3.7,191,0.820805
3,Raglan Road Irish Pub and Restaurant,Orlando,Irish,4,4.3,782,0.812483
4,Jon's head Grill,New Delhi,"Mexican, Italian, American",3,3.9,105,0.801221


In [15]:
# Demo 2: synthetic/manual profile
manual_profile = {
    'Country Code': 162,
    'City': 'Makati City',
    'Primary Cuisine': 'Japanese',
    'Average Cost for two': 1800,
    'Currency': 'Botswana Pula(P)',
    'Has Table booking': 1,
    'Has Online delivery': 0,
    'Is delivering now': 0,
    'Price range': 3,
    'Votes': 250
}

demo2 = run_flow(manual_profile=manual_profile, top_n=5)
print('Source:', demo2['source_restaurant'])
print('Predicted rating:', demo2['predicted_rating'])
print('Predicted cuisines:', demo2['predicted_cuisine_labels'])
display(demo2['top_recommendations'])

Source: None
Predicted rating: 3.617
Predicted cuisines: ['American', 'Asian', 'Cafe', 'Desserts', 'European', 'French', 'Indian', 'Italian', 'Japanese', 'Korean', 'Mexican', 'Middle Eastern', 'Seafood']


,Restaurant Name,City,Cuisines,Price range,Aggregate rating,Votes,score
0,Sushi Masa,Jakarta,"Sushi, Japanese",3,4.9,605,0.998
1,Le Petit Souffle,Makati City,"French, Japanese, Desserts",3,4.8,314,0.996
2,Roka,London,"Japanese, Sushi",3,4.6,436,0.992
3,Bone Daddies,London,"Ramen, Japanese",3,4.6,418,0.992
4,Izakaya Kikufuji,Makati City,Japanese,3,4.5,591,0.990


In [16]:
# Cross-task metric summary (quick checkpoint in integrated notebook)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=SEED)
rating_model_ckpt = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_r),
        ('num', 'passthrough', num_r)
    ])),
    ('model', DecisionTreeRegressor(random_state=SEED, max_depth=12, min_samples_leaf=5))
])
rating_model_ckpt.fit(Xr_train, yr_train)
yr_pred = rating_model_ckpt.predict(Xr_test)

Xc_train, Xc_test, Yc_train, Yc_test = train_test_split(Xc, Yc, test_size=0.2, random_state=SEED)
cuisine_model_ckpt = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_c),
        ('num', 'passthrough', num_c)
    ])),
    ('clf', OneVsRestClassifier(LogisticRegression(max_iter=2000, class_weight='balanced')))
])
cuisine_model_ckpt.fit(Xc_train, Yc_train)
Yc_pred = cuisine_model_ckpt.predict(Xc_test)

p_w, r_w, f1_w, _ = precision_recall_fscore_support(Yc_test, Yc_pred, average='weighted', zero_division=0)
subset_acc = accuracy_score(Yc_test, Yc_pred)

metrics_summary = pd.DataFrame([
    {'Task': 'Task 1 Regression', 'Metric': 'MSE', 'Value': mean_squared_error(yr_test, yr_pred)},
    {'Task': 'Task 1 Regression', 'Metric': 'R2', 'Value': r2_score(yr_test, yr_pred)},
    {'Task': 'Task 3 Multi-label', 'Metric': 'Weighted Precision', 'Value': p_w},
    {'Task': 'Task 3 Multi-label', 'Metric': 'Weighted Recall', 'Value': r_w},
    {'Task': 'Task 3 Multi-label', 'Metric': 'Weighted F1', 'Value': f1_w},
    {'Task': 'Task 3 Multi-label', 'Metric': 'Subset Accuracy', 'Value': subset_acc},
    {'Task': 'Task 2 Recommender', 'Metric': 'Rule Check', 'Value': 'Self-exclusion + unrated filter + votes tiebreak enabled'}
])
display(metrics_summary)

,Task,Metric,Value
0,Task 1 Regression,MSE,0.112555
1,Task 1 Regression,R2,0.950549
2,Task 3 Multi-label,Weighted Precision,0.260563
3,Task 3 Multi-label,Weighted Recall,0.700368
4,Task 3 Multi-label,Weighted F1,0.343782
5,Task 3 Multi-label,Subset Accuracy,0.0
6,Task 2 Recommender,Rule Check,Self-exclusion + unrated filter + votes tiebre...
